In [ ]:
# Install required libraries (Colab already has most)
!pip install matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------
# 1. Sample Dataset
# ---------------------------
input_texts = ["hi", "hello", "how are you", "i am fine", "thank you"]
target_texts = ["salut", "bonjour", "comment ca va", "je vais bien", "merci"]

# Add start and end tokens
target_texts = ["<start> " + txt + " <end>" for txt in target_texts]

# ---------------------------
# 2. Tokenization
# ---------------------------
input_tokenizer = Tokenizer(filters='')
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)
input_seq = pad_sequences(input_seq, padding='post')

target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)
target_seq = target_tokenizer.texts_to_sequences(target_texts)
target_seq = pad_sequences(target_seq, padding='post')

input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

max_len_input = input_seq.shape[1]
max_len_target = target_seq.shape[1]

# ---------------------------
# 3. Attention Layer
# ---------------------------
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, hidden, enc_output):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        score = self.V(tf.nn.tanh(
            self.W1(enc_output) + self.W2(hidden_with_time_axis)
        ))

        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * enc_output
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

# ---------------------------
# 4. Encoder
# ---------------------------
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x)
        return output, state_h

# ---------------------------
# 5. Decoder
# ---------------------------
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)
        self.attention = BahdanauAttention(units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)

        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        output, state_h, _ = self.lstm(x)
        output = self.fc(output)

        return output, state_h, attention_weights

# ---------------------------
# 6. Model Parameters
# ---------------------------
embedding_dim = 64
units = 128
EPOCHS = 200

encoder = Encoder(input_vocab_size, embedding_dim, units)
decoder = Decoder(target_vocab_size, embedding_dim, units)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------------------
# 7. Training Loop
# ---------------------------
for epoch in range(EPOCHS):
    total_loss = 0

    for i in range(len(input_seq)):
        inp = tf.expand_dims(input_seq[i], 0)
        targ = tf.expand_dims(target_seq[i], 0)

        loss = 0

        with tf.GradientTape() as tape:
            enc_output, enc_hidden = encoder(inp)
            dec_hidden = enc_hidden

            dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']], 0)

            for t in range(1, targ.shape[1]):
                predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                loss += loss_object(targ[:, t], predictions[:, 0])

                dec_input = tf.expand_dims([targ[0][t]], 0)

        variables = encoder.trainable_variables + decoder.trainable_variables
        gradients = tape.gradient(loss, variables)
        optimizer.apply_gradients(zip(gradients, variables))

        total_loss += loss

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss {total_loss.numpy()}")

# ---------------------------
# 8. Prediction + Attention
# ---------------------------
def evaluate(sentence):
    sequence = input_tokenizer.texts_to_sequences([sentence])
    sequence = pad_sequences(sequence, maxlen=max_len_input, padding='post')

    enc_out, enc_hidden = encoder(sequence)
    dec_hidden = enc_hidden

    dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']], 0)

    result = []
    attention_plot = []

    for t in range(max_len_target):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)

        attention_plot.append(tf.reshape(attention_weights, (-1,)))

        predicted_id = tf.argmax(predictions[0][0]).numpy()
        word = target_tokenizer.index_word.get(predicted_id, '')

        if word == '<end>':
            break

        result.append(word)
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, attention_plot

# ---------------------------
# 9. Visualization
# ---------------------------
def plot_attention(attention, sentence, predicted_sentence):
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(1,1,1)

    attention = np.array(attention)
    ax.matshow(attention, cmap='viridis')

    fontdict = {'fontsize': 10}

    ax.set_xticklabels([''] + sentence.split(), fontdict=fontdict, rotation=90)
    ax.set_yticklabels([''] + predicted_sentence, fontdict=fontdict)

    plt.show()

# ---------------------------
# 10. Test
# ---------------------------
sentence = "hello"
result, attention = evaluate(sentence)

print("Input:", sentence)
print("Predicted translation:", " ".join(result))

plot_attention(attention, sentence, result)